# Notebook 3 — Model Training

This notebook trains six models of increasing complexity to predict 
startup exit outcomes (acquired or IPO by January 2014).

**Model ladder:**
1. **Majority Class Baseline** — always predicts no exit. Sets the floor.
2. **Logistic Regression** — linear model with L2 regularisation. 
3. **Random Forest** — ensemble of decision trees with GridSearchCV tuning. 
   Captures non-linear feature interactions.
4. **XGBoost** — gradient boosted trees. Each tree corrects the previous 
   tree's errors via gradient descent,  using  Lecture 5 
   SGD update rule w^(i+1) = w^i − α∇L.
5. **MLP Neural Network** — differentiable function approximator f(x,w) 
   trained via backpropagation and SGD. Three-configuration ablation study isolates optimiser and feature effects.
6. **Encoder-Only Transformer** —  Each tabular feature is one token,
   attention learns which features contextualise each other.

**Evaluation strategy:**
- As-of date: January 2014 (Crunchbase snapshot)
- Temporal split: Train 1995–2006 | Validation 2007 | Test 2008–2009
- Models 1–4: 5-fold stratified CV on X_train
- Models 5–6: trained on X_train, early stopping on X_val (2007)
- Test set (2008–2009) untouched — evaluated once in Notebook 4

**Class imbalance** (81.7% no exit vs 18.3% exit in training set) 
handled via class_weight='balanced' for sklearn models and 
scale_pos_weight for XGBoost.

In [76]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RandomizedSearchCV, cross_validate)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, roc_auc_score,
                              f1_score, confusion_matrix,
                              precision_score, recall_score)
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

DATA_PROCESSED = Path('../data/processed')
OUTPUTS        = Path('../outputs/metrics')
FIGURES        = Path('../outputs/figures')
OUTPUTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

### Train / Test Split and Cross-Validation Strategy

I used a **stratified random 80/20 split** to divide the cohort into 
training (n=11,014) and held-out test (n=2,754) sets. Stratification 
ensures both sets maintain the observed exit rate of 13.9%.

**Why stratified random rather than temporal split:**
A temporal split (train on earlier years, test on later years) would be 
ideal for deployment — mimicking a real PE analyst screening future companies. 
However, our cohort (founded 1995–2009) uses a fixed snapshot date of 
January 2014 for all features, meaning all companies are observed at the 
same point in time. The prediction task is cross-sectional ("which of these 
companies will eventually exit?") rather than strictly temporal 
("will companies founded after date X exit?"). A stratified random split 
is therefore appropriate and consistent with Arroyo et al. (2019) who use 
the same approach for VC investment prediction.

**Cross-validation:** Stratified 5-Fold CV on the training set is used 
for all model selection and hyperparameter tuning (Arroyo et al., 2019; 
Hastie et al., 2009). The test set is touched only once — for final 
evaluation in Notebook 4.

**Note on transformer evaluation:** Model 6 (Transformer) uses a 
separate 10% validation split carved from training data for early 
stopping, keeping the test set strictly held out. All other models 
use cross-validation AUC for model selection.

In [77]:
AS_OF_DATE = "2014-01-01"  # Crunchbase snapshot date

X = features[FEATURE_COLS]
y = features['label']  # exit = acquired or IPO by Jan 2014

# Temporal split by founding year
# Train: 1995–2006 (earlier market conditions)
# Val:   2007      (intermediate — for transformer early stopping)
# Test:  2008–2009 (held out, later cohort)
train_mask = features['founded_year'].between(1995, 2006)
val_mask   = features['founded_year'].between(2007, 2007)
test_mask  = features['founded_year'].between(2008, 2009)

X_train = X[train_mask]
X_val   = X[val_mask]
X_test  = X[test_mask]
y_train = y[train_mask]
y_val   = y[val_mask]
y_test  = y[test_mask]

# Stratified CV on training set for sklearn models
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Sanity checks — no overlap
assert len(set(X_train.index) & set(X_val.index))  == 0
assert len(set(X_train.index) & set(X_test.index)) == 0
assert len(set(X_val.index)   & set(X_test.index)) == 0

print(f"=== Temporal Split (As-of: {AS_OF_DATE}) ===")
print(f"Train: {len(X_train):,} | founded 1995–2006 | "
      f"exit rate: {y_train.mean()*100:.1f}%")
print(f"Val:   {len(X_val):,}   | founded 2007      | "
      f"exit rate: {y_val.mean()*100:.1f}%")
print(f"Test:  {len(X_test):,}  | founded 2008–2009 | "
      f"exit rate: {y_test.mean()*100:.1f}%")
print(f"\nTrain exits: {y_train.sum():,}")
print(f"Val exits:   {y_val.sum():,}")
print(f"Test exits:  {y_test.sum():,}")


=== Temporal Split (As-of: 2014-01-01) ===
Train: 7,739 | founded 1995–2006 | exit rate: 18.3%
Val:   1,906   | founded 2007      | exit rate: 10.4%
Test:  4,123  | founded 2008–2009 | exit rate: 7.4%

Train exits: 1,413
Val exits:   199
Test exits:  306


### Model 1 — Majority Class Baseline

The simplest possible model: always predicts 0 (no exit).
This sets the performance floor, so any useful model must beat this.
Expected AUC = 0.50 (equivalent to random guessing).

In [58]:
# ═══════════════════════════════════════════════════════════════
# MODEL 1 — Majority Class Baseline
# Always predicts 0 (no exit) — the most common class (86.1%)
# Purpose: sets the floor — any real model must beat this
# ═══════════════════════════════════════════════════════════════

dummy = DummyClassifier(strategy='most_frequent', random_state=42)

dummy_scores = cross_validate(
    dummy, X_train, y_train,
    cv=cv,
    scoring=['roc_auc', 'f1_macro'],
    return_train_score=False
)

print("=== Model 1: Majority Class Baseline ===")
print(f"CV AUC:      {dummy_scores['test_roc_auc'].mean():.3f} "
      f"± {dummy_scores['test_roc_auc'].std():.3f}")
print(f"CV F1 macro: {dummy_scores['test_f1_macro'].mean():.3f} "
      f"± {dummy_scores['test_f1_macro'].std():.3f}")

=== Model 1: Majority Class Baseline ===
CV AUC:      0.500 ± 0.000
CV F1 macro: 0.450 ± 0.000


### Model 2 — Logistic Regression

Linear model with L2 regularisation. Fast, interpretable, and a strong 
baseline for binary classification. 

L2 regularisation (parameter C) penalises large coefficients, preventing 
overfitting. We tune C using cross-validation.


In [59]:
# Linear model with L2 regularisation
# Pipeline: scale features → fit logistic regression
# class_weight='balanced' handles class imbalance automatically
# ═══════════════════════════════════════════════════════════════

lr_pipeline = Pipeline([
    # Step 1: scale all features to same range
    # Logistic regression is sensitive to feature magnitude
    # Without scaling, total_funding_usd (millions) dominates
    # over is_us (0 or 1)
    ('scaler', StandardScaler()),

    # Step 2: fit logistic regression
    # C = regularisation strength (smaller = more regularisation)
    # class_weight='balanced' = up-weights minority exit class
    # max_iter=1000 = enough iterations to converge
    ('clf', LogisticRegression(
        C=1.0,
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])

lr_scores = cross_validate(
    lr_pipeline, X_train, y_train,
    cv=cv,
    scoring=['roc_auc', 'f1_macro'],
    return_train_score=False
)

print("=== Model 2: Logistic Regression ===")
print(f"CV AUC:      {lr_scores['test_roc_auc'].mean():.3f} "
      f"± {lr_scores['test_roc_auc'].std():.3f}")
print(f"CV F1 macro: {lr_scores['test_f1_macro'].mean():.3f} "
      f"± {lr_scores['test_f1_macro'].std():.3f}")
print(f"\nImprovement over baseline:")
print(f"AUC: +{lr_scores['test_roc_auc'].mean() - 0.500:.3f}")
print(f"F1:  +{lr_scores['test_f1_macro'].mean() - 0.463:.3f}")

=== Model 2: Logistic Regression ===
CV AUC:      0.770 ± 0.013
CV F1 macro: 0.636 ± 0.010

Improvement over baseline:
AUC: +0.270
F1:  +0.173


### Model 3 — Random Forest

Ensemble of decision trees that captures non-linear interactions between 
features. I used GridSearchCV to tests ALL combinations, evaluated using 5-fold cross validation.

Key hyperparameters tuned:
- n_estimators: number of trees (more = more stable but slower)
- max_depth: how deep each tree grows (deeper = more complex)
- min_samples_leaf: minimum samples per leaf (higher = less overfitting)

In [67]:
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

# ═══════════════════════════════════════════════════════════════
# MODEL 3 — Random Forest with GridSearchCV
# GridSearchCV tests ALL combinations in the grid
# Scored on AUC — our primary metric
# ═══════════════════════════════════════════════════════════════

param_grid = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [5, 10, 15, 20, None],
    'min_samples_leaf': [1, 5, 10, 20],
    'max_features':     ['sqrt', 'log2']
}

rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0,
    return_train_score=False
)

rf_grid.fit(X_train, y_train)

rf_auc_mean = rf_grid.best_score_
rf_auc_std  = rf_grid.cv_results_['std_test_score'][rf_grid.best_index_]

# F1 only for descriptive reporting
rf_f1_scores = cross_validate(
    rf_grid.best_estimator_,
    X_train, y_train,
    cv=cv,
    scoring=['f1_macro'],
    return_train_score=False
)

rf_f1_mean = rf_f1_scores['test_f1_macro'].mean()
rf_f1_std  = rf_f1_scores['test_f1_macro'].std()

print("=== Model 3: Random Forest (Grid Search) ===")
print(f"Best params:  {rf_grid.best_params_}")
print(f"CV AUC:       {rf_auc_mean:.3f} ± {rf_auc_std:.3f}")
print(f"CV F1 macro:  {rf_f1_mean:.3f} ± {rf_f1_std:.3f}")

print(f"\nImprovement over logistic regression:")
print(f"AUC: +{rf_auc_mean - lr_scores['test_roc_auc'].mean():.3f}")

=== Model 3: Random Forest (Grid Search) ===
Best params:  {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 500}
CV AUC:       0.792 ± 0.011
CV F1 macro:  0.646 ± 0.012

Improvement over logistic regression:
AUC: +0.022


### Model 4 — XGBoost (Gradient Boosted Trees)

XGBoost (eXtreme Gradient Boosting) extends Random Forest by building trees 
**sequentially** rather than independently. Each new tree fits the negative 
gradient of the loss function, therefore directly correcting the errors of all 
previous trees.

**Connection to Lecture 5 (Value Function Approximation):**
XGBoost optimises a differentiable loss function using gradient descent:

    w^(i+1) = w^i - α × ∇L(w^i)

where each successive tree represents one gradient step. This is the same 
iterative SGD update rule discussed in Lecture 5, applied to an ensemble 
of decision trees rather than neural network weights.

**Why XGBoost typically outperforms Random Forest:**
- Random Forest averages all trees equally (bagging)
- XGBoost weights each tree by its gradient contribution (boosting)
- Built-in L1/L2 regularisation prevents overfitting
- Handles class imbalance via scale_pos_weight parameter

**Key hyperparameters tuned:**
- n_estimators: number of boosting rounds
- max_depth: maximum tree depth
- learning_rate: step size (α in SGD analogy)
- subsample: fraction of training data per tree
- scale_pos_weight: handles class imbalance (ratio of negatives to positives)

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, cross_validate
import numpy as np
# Class imbalance weight — ratio of negatives to positives
# Equivalent to class_weight='balanced' in sklearn models
# scale_pos_weight = 11850/1413 ≈ 5.5 on temporal train set
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

# Hyperparameter search space
xgb_params = {
    'n_estimators':     [100, 200, 300, 500],  # boosting rounds
    'max_depth':        [3, 4, 5, 6],           # tree depth
    'learning_rate':    [0.01, 0.05, 0.1, 0.2], # step size α
    'subsample':        [0.6, 0.8, 1.0],         # row subsampling
    'colsample_bytree': [0.6, 0.8, 1.0]          # feature subsampling
}

xgb = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    # Note: use_label_encoder removed — obsolete in XGBoost 3.x
    eval_metric='logloss',  # internal eval metric during training
    random_state=42,
    n_jobs=-1
)

# RandomizedSearchCV — 20 random combinations × 5-fold CV
# Optimises AUC — our primary metric
xgb_grid = GridSearchCV(
    estimator=xgb,
    param_grid=xgb_params,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0,
    return_train_score=False
)

xgb_grid.fit(X_train, y_train)

# ── AUC: from search directly (no double-dipping) ─────────────
xgb_auc_mean = xgb_grid.best_score_
xgb_auc_std  = xgb_grid.cv_results_[
    'std_test_score'][xgb_grid.best_index_]

# ── F1: separate cross_validate (reporting only, not selection) ─
xgb_f1_scores = cross_validate(
    xgb_grid.best_estimator_,
    X_train, y_train,
    cv=cv,
    scoring=['f1_macro'],
    return_train_score=False
)
xgb_f1_mean = xgb_f1_scores['test_f1_macro'].mean()
xgb_f1_std  = xgb_f1_scores['test_f1_macro'].std()

# Store for comparison table
xgb_scores = {
    'test_roc_auc':  np.array([xgb_auc_mean]),
    'test_f1_macro': np.array([xgb_f1_mean])
}

print(f"Best params: {xgb_grid.best_params_}")
print(f"\n=== Model 4: XGBoost ===")
print(f"CV AUC:      {xgb_auc_mean:.3f} ± {xgb_auc_std:.3f}"
     )
print(f"CV F1 macro: {xgb_f1_mean:.3f} ± {xgb_f1_std:.3f}"
     )
print(f"\nImprovement over Random Forest:")
print(f"AUC: +{xgb_auc_mean - rf_auc_mean:.3f}")

scale_pos_weight: 4.5
Best params: {'colsample_bytree': 0.6, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 500, 'subsample': 0.6}

=== Model 4: XGBoost ===
CV AUC:      0.802 ± 0.009  (RandomizedSearchCV — used for selection)
CV F1 macro: 0.664 ± 0.010  (cross_validate — reporting only)

Improvement over Random Forest:
AUC: +0.010


### Model 5 — Neural Network Function Approximator

The implementation of a neural network as a differentiable function approximator f(x, w), where x is the 216-dimensional company feature vector and w are the learnable parameters 
(weights and biases).

As stated in Lecture 5: "By choosing approximators which are differentiable 
in w, we ensure that we can use stochastic gradient descent to find the 
right w. For neural networks, we use backprop to compute the gradient."

The SGD parameter update from Lecture 5 applies directly:

    w^(i+1) = w^i − α × ∇_w l(f(x, w^i), y)

where l is binary cross-entropy loss, α is the learning rate, and the 
gradient ∇_w is computed via backpropagation through the network layers.

**Architecture:**
- Input layer: 216 features (company state representation)
- Hidden layer 1: 128 neurons, ReLU activation
- Hidden layer 2: 64 neurons, ReLU activation
- Output layer: sigmoid → exit probability ∈ [0,1]

**Ablation study — optimiser and feature set:**

Three configurations are tested to isolate two effects:
1. Effect of optimiser: SGD (Lecture 5) vs Adam (Kingma & Ba, 2014)
2. Effect of TF-IDF features: sparse text features vs tabular only

| Config | Optimiser | Features |
|--------|-----------|----------|
| 5a | SGD | All 216 (includes TF-IDF) |
| 5b | Adam | All 216 (includes TF-IDF) |
| 5c | SGD | 63 tabular only (no TF-IDF) |

**SGD** directly implements the Lecture 5 update rule with fixed α.
**Adam** (Kingma & Ba, 2014) extends SGD with adaptive per-parameter 
learning rates which is better suited for sparse inputs. Comparing 5a vs 5b isolates optimiser effect. Comparing 5a vs 5c isolates feature set effect.


In [ ]:
# Three configurations testing optimiser and feature set effects

# Define tabular-only feature set (no TF-IDF)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# Define tabular-only feature set (no TF-IDF)
TEXT_COLS    = [c for c in FEATURE_COLS if 'tfidf_' in c or
                c in ['desc_word_count', 'desc_char_count',
                      'has_description']]
TABULAR_COLS = [c for c in FEATURE_COLS if c not in TEXT_COLS]

print(f"All features:     {len(FEATURE_COLS)}")
print(f"Tabular only:     {len(TABULAR_COLS)}")
print(f"Text features:    {len(TEXT_COLS)}")

def train_mlp(X_tr, y_tr, X_v, y_v, solver, features, label):
    """
    Train MLP on X_tr, use X_v for early stopping.
    Returns val AUC. Same protocol as transformer.
    """
    # Scale features — fit on training only, transform val
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr[features])
    X_v_s  = scaler.transform(X_v[features])

    if solver == 'sgd':
        clf = MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation='relu',
            solver='sgd',
            learning_rate='adaptive',
            learning_rate_init=0.01,
            momentum=0.9,
            alpha=0.001,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=15,
            max_iter=500,
            random_state=42
        )
    else:  # adam
        clf = MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation='relu',
            solver='adam',
            learning_rate_init=0.001,
            alpha=0.001,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            max_iter=500,
            random_state=42
        )

    # Fit on X_train (1995-2006)
    clf.fit(X_tr_s, y_tr)

    # Evaluate on X_val (2007) 
    val_probs = clf.predict_proba(X_v_s)[:, 1]
    val_auc   = roc_auc_score(y_v, val_probs)

    print(f"\n=== {label} ===")
    print(f"Val AUC (2007): {val_auc:.3f}")
    print(f"Iterations:     {clf.n_iter_}")
    print(f"Features used:  {len(features)}")

    return clf, scaler, val_auc

# ── 5a: SGD + all 216 features ────────────────────────────────
mlp_5a, scaler_5a, auc_5a = train_mlp(
    X_train, y_train, X_val, y_val,
    solver='sgd',
    features=FEATURE_COLS,
    label='5a: MLP SGD, all 216 features'
)

# ── 5b: Adam + all 216 features ───────────────────────────────
mlp_5b, scaler_5b, auc_5b = train_mlp(
    X_train, y_train, X_val, y_val,
    solver='adam',
    features=FEATURE_COLS,
    label='5b: MLP Adam, all 216 features'
)

# ── 5c: SGD + 63 tabular features only ────────────────────────
mlp_5c, scaler_5c, auc_5c = train_mlp(
    X_train, y_train, X_val, y_val,
    solver='sgd',
    features=TABULAR_COLS,
    label='5c: MLP SGD, 63 tabular features only'
)

# ── Ablation summary ──────────────────────────────────────────
print("\n=== Ablation Summary ===")
print(f"{'Config':<35} {'Val AUC (2007)':>14}")
print("-" * 52)
print(f"{'5a: SGD + all features':<35} {auc_5a:>14.3f}")
print(f"{'5b: Adam + all features':<35} {auc_5b:>14.3f}")
print(f"{'5c: SGD + tabular only':<35} {auc_5c:>14.3f}")
print(f"\nOptimiser effect (5a vs 5b): "
      f"+{auc_5b - auc_5a:.3f} AUC for Adam")
print(f"Feature effect (5a vs 5c):  "
      f"+{auc_5c - auc_5a:.3f} AUC removing TF-IDF")

# Best MLP config
best_mlp_auc = max(auc_5a, auc_5b, auc_5c)
if best_mlp_auc == auc_5c:
    print("\nBest MLP: 5c (SGD, tabular only)")
    best_mlp     = mlp_5c
    best_scaler  = scaler_5c
    best_mlp_features = TABULAR_COLS
elif best_mlp_auc == auc_5b:
    print("\nBest MLP: 5b (Adam, all features)")
    best_mlp     = mlp_5b
    best_scaler  = scaler_5b
    best_mlp_features = FEATURE_COLS
else:
    print("\nBest MLP: 5a (SGD, all features)")
    best_mlp     = mlp_5a
    best_scaler  = scaler_5a
    best_mlp_features = FEATURE_COLS

print(f"Best val AUC: {best_mlp_auc:.3f}")


All features:     216
Tabular only:     63
Text features:    153

=== 5a: MLP SGD, all 216 features ===
Val AUC (2007): 0.701
Iterations:     118
Features used:  216

=== 5b: MLP Adam, all 216 features ===
Val AUC (2007): 0.715
Iterations:     24
Features used:  216

=== 5c: MLP SGD, 63 tabular features only ===
Val AUC (2007): 0.754
Iterations:     117
Features used:  63

=== Ablation Summary ===
Config                              Val AUC (2007)
----------------------------------------------------
5a: SGD + all features                       0.701
5b: Adam + all features                      0.715
5c: SGD + tabular only                       0.754

Optimiser effect (5a vs 5b): +0.014 AUC for Adam
Feature effect (5a vs 5c):  +0.053 AUC removing TF-IDF

Best MLP: 5c (SGD, tabular only)
Best val AUC: 0.754

Note: val AUC uses X_val (2007) — same protocol as


### Model 6 — Encoder-Only Transformer (Lectures 9–10)

Following Lectures 9–10, we implement an encoder-only transformer for 
startup exit classification. Lecture 10 states:

"An encoder-only transformer takes in a sequence of tokens, embeds them 
into an embedding space, adds positional encoding, feeds through N blocks 
of multi-headed attention with skip connections and layer normalisation, 
then adds a fully-connected output layer. Encoder-only architectures are 
often used to classify."

**Adaptation to tabular data:**
Each of the 63 tabular features is treated as one token. The attention 
mechanism learns which features attend to each other — for example, 
whether n_rounds and funding_stage_score co-attend strongly, meaning 
their combination is more predictive than either alone.

This mirrors the contextual attention example from Lecture 9 where 
"interest" attends to "loan" to resolve meaning — here total_funding 
attends to reached_series_b to resolve exit probability.

**Architecture (Lecture 10 encoder structure):**
- Input embedding: linear projection of each feature → d_model=32
- Positional encoding: learnable (features have no natural order)
- N=2 encoder blocks, each containing:
  1. Multi-head self-attention (4 heads) + skip connection
  2. Layer normalisation
  3. Feed-forward layer + skip connection
  4. Layer normalisation
- Output: mean pooling → fully-connected → sigmoid

**Training:** Adam optimiser (Kingma & Ba, 2014) with early stopping.
Uses same 63 tabular features as MLP 5c for fair comparison.

In [69]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score
torch.manual_seed(42)

# ═══════════════════════════════════════════════════════════════
# MODEL 6 — Encoder-Only Transformer (Lectures 9–10, Problem Set 9)
#
# Key course connections:
# - Problem Set 9: scaled dot-product attention Q,K,V computation
# - Lecture 9: embeddings, multi-head attention
# - Lecture 10: encoder architecture (Add&Norm, FFN, skip connections)
# - Lecture 5: SGD/Adam training loop, backpropagation
# ═══════════════════════════════════════════════════════════════

# ── Prepare data ──────────────────────────────────────────────
# Use 63 tabular features — same as MLP 5c for fair comparison
# TF-IDF excluded: ablation study showed it dilutes attention
scaler_t    = StandardScaler()
X_train_tab = scaler_t.fit_transform(X_train[TABULAR_COLS])
X_val_tab   = scaler_t.transform(X_val[TABULAR_COLS])
X_test_tab  = scaler_t.transform(X_test[TABULAR_COLS])

# Shape: (n_samples, n_features, 1)
# Each feature = one token with embedding dimension = 1
# Matches Problem Set 9 layout: E ∈ R^(dT × dE) where dE=1 initially
X_tr = torch.FloatTensor(X_train_tab).unsqueeze(-1)
X_va = torch.FloatTensor(X_val_tab).unsqueeze(-1)
X_te = torch.FloatTensor(X_test_tab).unsqueeze(-1)
y_tr = torch.FloatTensor(y_train.values)
y_te = torch.FloatTensor(y_test.values)
y_va = torch.FloatTensor(y_val.values)

# ──  Scaled Dot-Product Attention ──
class ScaledDotProductAttention(nn.Module):
    """
    Implements exactly the attention mechanism from Problem Set 9 cheat sheet
    (row-layout version, page 3):

    Given E ∈ R^(dT × dE):
      Q = E @ W_Q  ∈ R^(dT × dQ)   — what each token is looking for
      K = E @ W_K  ∈ R^(dT × dQ)   — what each token is offering
      V = E @ W_V  ∈ R^(dT × dE)   — what each token will contribute

      S = (1/√dQ) * Q @ K^T  ∈ R^(dT × dT)
      S_ij = Q_i · K_j = degree to which query token i matches key token j

      A = softmax_row-wise(S)  — rows sum to 1
      A_ij = exp(S_ij) / Σ_j' exp(S_ij')

      O = A @ V  ∈ R^(dT × dE)
      O_i = weighted sum of value vectors, weights given by row i of A
    """
    def __init__(self, d_model, d_q):
        super().__init__()
        # W_Q, W_K ∈ R^(dE × dQ) — project embeddings to query/key space
        self.W_Q = nn.Linear(d_model, d_q, bias=False)
        self.W_K = nn.Linear(d_model, d_q, bias=False)
        # W_V ∈ R^(dE × dE) — project embeddings to value space
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.d_q = d_q

    def forward(self, E, mask=None):
        # E shape: (batch, dT, dE)

        # Problem Set 9: Q = E @ W_Q, K = E @ W_K, V = E @ W_V
        Q = self.W_Q(E)                    # (batch, dT, dQ)
        K = self.W_K(E)                    # (batch, dT, dQ)
        V = self.W_V(E)                    # (batch, dT, dE)

        # S = (1/√dQ) * Q @ K^T  ∈ R^(dT × dT)
        # S_ij = how much query token i matches key token j
        S = torch.bmm(Q, K.transpose(-2, -1)) / (self.d_q ** 0.5)

        # Causal Mask
        if mask is not None:
            S = S + mask

        # A = softmax_row-wise(S) — rows sum to 1 
        # A_ij = probability that token i should attend to token j
        A = F.softmax(S, dim=-1)

        # O = A @ V  (Problem Set 9: O = AV)
        # Each token's output = weighted sum of all value vectors
        O = torch.bmm(A, V)

        return O, A


# ── Multi-Head Attention (Lecture 9) ──────────────────────────
class MultiHeadAttention(nn.Module):
    """
    Lecture 9: "Transformers typically work with multi-head attention
    where the embeddings go through various attention heads at once in
    parallel. The intuition is that different heads learn different ways
    in which one token influences the meaning of another."

    Each head learns a different contextual relationship:
    - Head 1 might focus on recency signals
    - Head 2 might focus on funding size signals
    - Head 3 might focus on stage progression signals
    - Head 4 might focus on team/education signals

    This is analogous to CNN filters (Lecture 8) — each filter detects
    a different pattern. Each attention head detects a different
    feature interaction pattern.
    """
    def __init__(self, d_model, d_q, n_heads=4):
        super().__init__()
        # n_heads independent attention heads (Problem Set 9 × n_heads)
        self.heads = nn.ModuleList([
            ScaledDotProductAttention(d_model, d_q)
            for _ in range(n_heads)
        ])
        # Project concatenated head outputs back to d_model
        # Lecture 9: "embedding updates of each head are then added"
        self.W_out = nn.Linear(d_model * n_heads, d_model, bias=False)
        self.n_heads = n_heads

    def forward(self, E, mask=None):
        head_outputs = []
        all_weights  = []

        # Each head independently computes Q, K, V and attention
        for head in self.heads:
            O, A = head(E, mask)
            head_outputs.append(O)
            all_weights.append(A)

        # Concatenate outputs from all heads → project to d_model
        O_cat = torch.cat(head_outputs, dim=-1)  # (batch, dT, d_model*n_heads)
        O_out = self.W_out(O_cat)                # (batch, dT, d_model)

        # Average attention weights across heads for interpretability
        avg_weights = torch.stack(all_weights).mean(dim=0)

        return O_out, avg_weights


# ── Encoder-Only Transformer (Lecture 10) ─────────────────────
class StartupTransformer(nn.Module):
    """
    Encoder-only Transformer for startup exit classification.
    Lecture 10: "An encoder-only transformer takes in a sequence of
    tokens, embeds them, adds positional encoding, feeds through N
    blocks of multi-headed attention with skip connections and layer
    normalisation, then adds a fully-connected output layer.
    Encoder-only architectures are often used to classify."

    Design choices:
    - d_model=16: reduced from 32 to limit parameters (~9k total)
      appropriate for 11,014 training samples
    - 4 heads: Lecture 9 justification for diverse attention patterns
    - No sigmoid in classifier: BCEWithLogitsLoss applies sigmoid
      internally — double sigmoid would squash gradients
    """
    def __init__(self, n_features, d_model=16, d_q=8,
                 n_heads=4, num_layers=2, d_ff=32, dropout=0.1):
        super().__init__()
        self.n_features = n_features

        # Step 1: Input Embedding (Lecture 9)
        # "multiply each token with an embedding matrix"
        self.embedding = nn.Linear(1, d_model)

        # Step 2: Positional Encoding (Lecture 10)
        # Learnable — features have no natural order
        self.pos_encoding = nn.Embedding(n_features, d_model)

        # Step 3: N Encoder Blocks (Lecture 10)
        # Each: MultiHeadAttention + Add&Norm + FFN + Add&Norm
        self.attention_layers = nn.ModuleList([
            MultiHeadAttention(d_model, d_q, n_heads)
            for _ in range(num_layers)
        ])
        self.norm1 = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(num_layers)
        ])
        self.ffn = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.ReLU(),
                nn.Linear(d_ff, d_model)
            ) for _ in range(num_layers)
        ])
        self.norm2 = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)

        # Step 4: Classification Head (Lecture 10)
        # NO sigmoid — BCEWithLogitsLoss applies sigmoid internally
        # Avoids double sigmoid which squashes gradients
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1)
            # No sigmoid here
        )

        self.last_attention = None

    def forward(self, x, use_causal_mask=False):
        batch_size, n_features, _ = x.shape

        # Step 1: embed each feature token
        E = self.embedding(x)

        # Step 2: add positional encoding
        positions = torch.arange(n_features, device=x.device)
        E = E + self.pos_encoding(positions)

        # Step 3: N encoder blocks
        for i in range(len(self.attention_layers)):
            mask = None
            if use_causal_mask:
                mask = torch.triu(
                    torch.ones(n_features, n_features) * float('-inf'),
                    diagonal=1
                ).to(x.device)

            # Attention: O = softmax(QK^T/√dQ) @ V
            attn_out, attn_weights = self.attention_layers[i](E, mask)
            self.last_attention = attn_weights

            # Add & Norm (Lecture 10, Lecture 8 skip connections)
            E = self.norm1[i](E + self.dropout(attn_out))

            # FFN + Add & Norm
            E = self.norm2[i](E + self.dropout(self.ffn[i](E)))

        # Step 4: mean pooling → classification head
        return self.classifier(E.mean(dim=1)).squeeze(-1)


# ── Training setup ────────────────────────────────────────────
n_features = X_train_tab.shape[1]
model      = StartupTransformer(n_features=n_features)

print("=== Model 6: Encoder-Only Transformer ===")
print(f"Input:      {n_features} tokens, dim=1 each")
print(f"Embedding:  1 → d_model=16 (Lecture 9)")
print(f"Encoder:    2 × (4-head attention + Add&Norm + FFN + Add&Norm)")
print(f"Attention:  d_q=8, scaled dot-product (Problem Set 9)")
print(f"Output:     mean pool → 16→16→1 (no sigmoid, Lecture 10)")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Note: no sigmoid in classifier — BCEWithLogitsLoss")
print(f"applies sigmoid internally (avoids double sigmoid)")

optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

pos_weight = torch.tensor(
    [(y_train == 0).sum() / (y_train == 1).sum()],
    dtype=torch.float
)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

train_dataset = TensorDataset(X_tr, y_tr)
train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True)

# ── Training loop ─────────────────────────────────────────────
print("\nTraining...")
best_auc   = 0
patience   = 15
no_improve = 0

for epoch in range(100):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()    # backprop (Lecture 5)
        optimizer.step()   # w = w - α∇L (Lecture 5)

    model.eval()
    with torch.no_grad():
        # Apply sigmoid manually — classifier outputs raw logits
        val_logits = model(X_va)
        val_preds  = torch.sigmoid(val_logits).numpy()
    auc = roc_auc_score(y_va , val_preds)

    if auc > best_auc:
        best_auc   = auc
        no_improve = 0
        torch.save(model.state_dict(), 'best_transformer.pt')
    else:
        no_improve += 1

    if no_improve >= patience:
        print(f"  Early stopping at epoch {epoch+1}")
        break

    if (epoch+1) % 20 == 0:
        print(f"  Epoch {epoch+1}/100 — Val AUC: {auc:.3f}")

# Load best and get final predictions
model.load_state_dict(torch.load('best_transformer.pt'))
model.eval()
with torch.no_grad():
    transformer_preds = torch.sigmoid(model(X_te)).numpy()

transformer_auc = roc_auc_score(y_test, transformer_preds)
print(f"\nBest Val AUC:  {best_auc:.3f} ")
print(f"Test AUC:      {transformer_auc:.3f} ")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ── Attention Analysis (Problem Set 9, Q3) ────────────────────
print("\n=== Attention Analysis (Problem Set 9, Q3) ===")
print("S_ij = degree to which query token i matches key token j")
print("A_ij = softmax(S)_ij = attention probability (rows sum to 1)")

exit_mask = (y_test == 1).values
X_exit    = X_te[exit_mask][:20]

model.eval()
with torch.no_grad():
    _ = model(X_exit)

attn = model.last_attention.mean(dim=0).numpy()

# Top attended pairs
rows, cols = np.triu_indices(len(TABULAR_COLS), k=1)
pairs = [(attn[r,c], TABULAR_COLS[r], TABULAR_COLS[c])
         for r,c in zip(rows, cols)]
pairs.sort(reverse=True)

print(f"\n{'Feature i (query)':<32} {'Feature j (key)':<32} {'A_ij':>6}")
print("-" * 74)
for score, f1, f2 in pairs[:10]:
    print(f"{f1:<32} {f2:<32} {score:>6.3f}")

# Most attended-to key
key_totals  = attn.sum(axis=0)
top_key_idx = key_totals.argmax()
print(f"\nMost attended-to feature (key): '{TABULAR_COLS[top_key_idx]}'")
print(f"Total attention received: {key_totals[top_key_idx]:.3f} / 63")
print(f"= {key_totals[top_key_idx]/63*100:.1f}% of all attention")
print(f"\nInterpretation (Lecture 9): analogous to 'interest' attending")
print(f"to 'loan' — most features contextualise their exit signal")
print(f"by attending to '{TABULAR_COLS[top_key_idx]}'")

=== Model 6: Encoder-Only Transformer ===
Input:      63 tokens, dim=1 each
Embedding:  1 → d_model=16 (Lecture 9)
Encoder:    2 × (4-head attention + Add&Norm + FFN + Add&Norm)
Attention:  d_q=8, scaled dot-product (Problem Set 9)
Output:     mean pool → 16→16→1 (no sigmoid, Lecture 10)
Parameters: 9,745
Note: no sigmoid in classifier — BCEWithLogitsLoss
applies sigmoid internally (avoids double sigmoid)

Training...
  Epoch 20/100 — Val AUC: 0.773
  Early stopping at epoch 28

Best Val AUC:  0.776 
Test AUC:      0.774 
Parameters: 9,745

=== Attention Analysis (Problem Set 9, Q3) ===
S_ij = degree to which query token i matches key token j
A_ij = softmax(S)_ij = attention probability (rows sum to 1)

Feature i (query)                Feature j (key)                    A_ij
--------------------------------------------------------------------------
funding_round_type_post-ipo      sector_advertising                0.058
funding_round_type_post-ipo      first_funding_age                

MODEL COMPARISON

In [ ]:

# Evaluation protocol
# Models 1–4: 5-fold stratified CV AUC on X_train (1995–2006)
# Models 5–6: Val AUC on X_val (2007) — temporal holdout
#
# Note: CV AUC and Val AUC use different protocols and are
# not directly comparable here. Final fair comparison of all
# models on the same held-out test set (2008–2009) is in
# Notebook 4.
# ═══════════════════════════════════════════════════════════════

print("=== MODEL COMPARISON — Notebook 3 ===")
print(f"{'Model':<45} {'AUC':>8} {'Protocol'}")
print("-" * 75)

# Models 1-4: CV AUC on X_train
print(f"{'Majority Class Baseline':<45} "
      f"{dummy_scores['test_roc_auc'].mean():>8.3f}  CV on train")
print(f"{'Logistic Regression':<45} "
      f"{lr_scores['test_roc_auc'].mean():>8.3f}  CV on train")
print(f"{'Random Forest':<45} "
      f"{rf_auc_mean:>8.3f}  CV on train")
print(f"{'XGBoost':<45} "
      f"{xgb_auc_mean:>8.3f}  CV on train ← best tree model")

# Models 5-6: Val AUC on X_val (2007)
print(f"{'MLP 5a (SGD, 216 features)':<45} "
      f"{auc_5a:>8.3f}  Val AUC (2007)")
print(f"{'MLP 5b (Adam, 216 features)':<45} "
      f"{auc_5b:>8.3f}  Val AUC (2007)")
print(f"{'MLP 5c (SGD, 63 tabular)':<45} "
      f"{auc_5c:>8.3f}  Val AUC (2007) ← best MLP")
print(f"{'Transformer 6a (63 tabular)':<45} "
      f"{transformer_auc:>8.3f}  Val→Test AUC (2007→2008-09)")

print(f"\n{'─'*75}")
print(f"Final evaluation on test set (2008-2009): Notebook 4")
print(f"All models evaluated on same held-out test set once.")

# Save
import pandas as pd
results = pd.DataFrame({
    'Model': [
        'Majority Class Baseline',
        'Logistic Regression',
        'Random Forest',
        'XGBoost',
        'MLP 5a (SGD, 216 features)',
        'MLP 5b (Adam, 216 features)',
        'MLP 5c (SGD, 63 tabular)',
        'Transformer 6a (63 tabular)'
    ],
    'AUC': [
        dummy_scores['test_roc_auc'].mean(),
        lr_scores['test_roc_auc'].mean(),
        rf_auc_mean,
        xgb_auc_mean,
        auc_5a,
        auc_5b,
        auc_5c,
        transformer_auc
    ],
    'Protocol': [
        'CV on train',
        'CV on train',
        'CV on train',
        'CV on train',
        'Val AUC 2007',
        'Val AUC 2007',
        'Val AUC 2007',
        'Test AUC 2008-09'
    ]
})

results.to_csv(OUTPUTS / 'model_comparison.csv', index=False)
print("\nSaved to outputs/metrics/model_comparison.csv")

=== MODEL COMPARISON — Notebook 3 ===
Model                                              AUC Protocol
---------------------------------------------------------------------------
Majority Class Baseline                          0.500  CV on train
Logistic Regression                              0.770  CV on train
Random Forest                                    0.792  CV on train
XGBoost                                          0.802  CV on train ← best tree model
MLP 5a (SGD, 216 features)                       0.701  Val AUC (2007)
MLP 5b (Adam, 216 features)                      0.715  Val AUC (2007)
MLP 5c (SGD, 63 tabular)                         0.754  Val AUC (2007) ← best MLP
Transformer 6a (63 tabular)                      0.774  Val→Test AUC (2007→2008-09)

───────────────────────────────────────────────────────────────────────────
Final evaluation on test set (2008-2009): Notebook 4
All models evaluated on same held-out test set once.

Saved to outputs/metrics/model_compariso